# SpiderNet tutorial: Pancancer example

This notebook demonstrates the **core SpiderNet workflow** on the **Pancancer dataset** and can be used as a **template for applying SpiderNet to a new dataset**.

## Core pipeline
1. **Edit the dataset setup cell**  
   Specify the **input paths**, **resource files**, and key dataset-specific settings.

2. **Check paths and AnnData fields**  
   Verify that all required files, metadata fields, and **AnnData structure** are correctly configured before running the pipeline.

3. **Preprocess the raw data**  
   Convert the input data into the **SpiderNet input format** needed for downstream model training.

4. **Train the model**  
   Run **SpiderNet training** to learn latent **meta-interactions (MIs)** and communication-related representations.

5. **Infer and export results**  
   Generate and save the main outputs, including **MI factors**, **gene / LR loadings**, and other downstream analysis files.

## Optional analyses
- **MI correlation**
- **LR pathway enrichment**
- **sender--receiver cell-type enrichment**

## Before you start

Under `DATA_ROOT`, keep an `adata/` folder with one `.h5ad` file per sample or batch.

Each `.h5ad` should contain:
- Spatial coordinates: `adata.obsm['spatial']`
- Sample IDs: `adata.obs[SAMPLE_ID_COL]`
- Pre-determined cell type labels: `adata.obs[CELL_TYPE_COL]`
- gene symbols in `adata.var_names` (for example `MYC`, `TP53`)



## 0. Dataset setup (edit this cell first)

Put the dataset-specific settings here. In most cases, this is the main cell you need to edit when switching to a new dataset.


In [1]:
from pathlib import Path
import json
import numpy as np

# ---------------------------------------------------------------------
# Dataset and output paths
# ---------------------------------------------------------------------
DATA_ROOT = Path("D:/SpiderNet/Data/Pancancer")      # Raw input data root
OUTPUT_ROOT = Path("D:/SpiderNet/Results/Pancancer") # Main result root

# The unified dataloader writes processed objects here.
# This folder is intentionally kept outside the model-training run directory.
PROCESSED_DATA_DIR = OUTPUT_ROOT / "ProcessedData"

# Folder under DATA_ROOT that stores the .h5ad files.
# For the full pan-cancer dataset, change this to "adata_entire" if needed.
ADATA_FOLDER_NAME = "adata"

# ---------------------------------------------------------------------
# Pancancer-specific AnnData fields
# ---------------------------------------------------------------------
SPECIES = "human"              # "human" or "mouse"
SAMPLE_COL = "SampleID"        # Constructed as <file_prefix>_subslice<subslice_id>
CELL_CLASS_COL = "celltype_final"
SPATIAL_KEY = "spatial"
PYG_EXTRA_OBS_FIELDS = {}

# Backward-compatible aliases used by some checks below.
SAMPLE_ID_COL = SAMPLE_COL
CELL_TYPE_COL = CELL_CLASS_COL

# ---------------------------------------------------------------------
# Preprocessing parameters
# ---------------------------------------------------------------------
N_HVG = 1000
N_HVG_LR = 2000
NUM_NEIGHBORS = 5

# Optional predefined lists.
# If LR_LIST_PATH is provided, LR activation/correlation filtering is skipped.
# If GENE_LIST_PATH is provided, HVG selection is skipped and this gene list is used.
LR_LIST_PATH = None
GENE_LIST_PATH = None

# Used only when LR_LIST_PATH is None.
# This follows the updated Pancancer MIdim-selection notebook.
LR_CORR_THRESHOLD = 0.2

# Set this to False if PROCESSED_DATA_DIR already contains a complete processed bundle.
RUN_PREPROCESSING = True

# ---------------------------------------------------------------------
# Training parameters
# ---------------------------------------------------------------------
DIM_ENVIR = 11
N_JOBS = 5
MAX_EPOCH = 20000
VERSION = "V1"

# Save the high-level user-facing config.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "DATA_ROOT": str(DATA_ROOT),
    "OUTPUT_ROOT": str(OUTPUT_ROOT),
    "PROCESSED_DATA_DIR": str(PROCESSED_DATA_DIR),
    "ADATA_FOLDER_NAME": ADATA_FOLDER_NAME,
    "SPECIES": SPECIES,
    "SAMPLE_COL": SAMPLE_COL,
    "CELL_CLASS_COL": CELL_CLASS_COL,
    "SPATIAL_KEY": SPATIAL_KEY,
    "N_HVG": N_HVG,
    "N_HVG_LR": N_HVG_LR,
    "NUM_NEIGHBORS": NUM_NEIGHBORS,
    "LR_LIST_PATH": str(LR_LIST_PATH) if LR_LIST_PATH is not None else None,
    "GENE_LIST_PATH": str(GENE_LIST_PATH) if GENE_LIST_PATH is not None else None,
    "LR_CORR_THRESHOLD": LR_CORR_THRESHOLD,
    "RUN_PREPROCESSING": RUN_PREPROCESSING,
    "DIM_ENVIR": DIM_ENVIR,
    "N_JOBS": N_JOBS,
    "MAX_EPOCH": MAX_EPOCH,
    "VERSION": VERSION,
}

config_path = OUTPUT_ROOT / "Pancancer_modeltraining_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print(f"Config saved to: {config_path}")


Config saved to: D:\SpiderNet\Results\Pancancer\Pancancer_modeltraining_config.json


## 1. Imports and device setup

Load SpiderNet utilities and choose CPU or CUDA.


In [2]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch

from SpiderNet.utils import *
from SpiderNet.config import *
from SpiderNet.dataloading_unified import (
    prepare_processed_bundle_unified,
    preview_lr_corr_distribution,
)
from SpiderNet.utils import get_default_cellchat_db, get_default_scseqcomm_db

cuda_available = torch.cuda.is_available()
device = "cuda" if cuda_available else "cpu"
print(f"Using device: {device}")


C:\Users\junji\miniconda3\envs\SpiderNet_env\Lib\site-packages\louvain\__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


Using device: cuda


## 2. Build paths from the dataset setup

This cell uses the values from the dataset setup cell above.


In [3]:
paths = PathConfig(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    species=SPECIES,
    version=VERSION,
)

paths


PathConfig(data_root=WindowsPath('D:/SpiderNet/Data/Pancancer'), output_root=WindowsPath('D:/SpiderNet/Results/Pancancer'), species='human', cellchat_db=WindowsPath('D:/SpiderNet/SpiderNet_proj/SpiderNet_Project/SpiderNet/resources/Human_LR_pairs_Cellchatdb.csv'), scseqcomm_db=WindowsPath('D:/SpiderNet/SpiderNet_proj/SpiderNet_Project/SpiderNet/resources/Human_LR_pairs_scSeqComm.csv'), version='V1')

In [4]:
CELLCHAT_DB = get_default_cellchat_db(species=SPECIES)
SCSEQCOMM_DB = get_default_scseqcomm_db(species=SPECIES)

def normalize_optional_path_local(path_str):
    if path_str is None:
        return None
    path_str = str(path_str).strip()
    if path_str.lower() in {"", "none", "null", "nan"}:
        return None
    return path_str

def pancancer_file_prefix(context):
    return Path(context["file_name"]).stem.split("_")[0]

def pancancer_per_file_hook(adata, context):
    """Match the original Pancancer loader while keeping raw .h5ad files unchanged."""
    sample_prefix = pancancer_file_prefix(context)

    if sp.issparse(adata.X):
        adata.X = adata.X.astype(np.float32)
    else:
        adata.X = np.asarray(adata.X, dtype=np.float32)

    adata.obs_names = [f"{sample_prefix}_{cellname}" for cellname in adata.obs_names]

    if "subslice_id" not in adata.obs.columns:
        raise KeyError(
            "Pancancer preprocessing expects adata.obs['subslice_id'] so that "
            "SampleID can be constructed as <file_prefix>_subslice<subslice_id>."
        )

    adata.obs["source_file_prefix"] = sample_prefix
    adata.obs[SAMPLE_COL] = [
        f"{sample_prefix}_subslice{subslice_id}"
        for subslice_id in adata.obs["subslice_id"].astype(str)
    ]

    if SPATIAL_KEY not in adata.obsm:
        raise KeyError(
            f"Pancancer preprocessing expects adata.obsm[{SPATIAL_KEY!r}] "
            "to contain spatial coordinates."
        )

    # Keep only the fields needed for SpiderNet preprocessing.
    adata.layers.clear()
    for key in ["X_pca", "X_umap"]:
        if key in adata.obsm:
            del adata.obsm[key]
    adata.obsm = {"spatial": np.asarray(adata.obsm[SPATIAL_KEY])}
    adata.varm.clear()
    adata.raw = None
    if adata.var.shape[1] > 0:
        adata.var = adata.var.iloc[:, []].copy()

    return adata

APPLY_HVG_SELECTION = normalize_optional_path_local(GENE_LIST_PATH) is None
APPLY_LR_CORR_FILTER = normalize_optional_path_local(LR_LIST_PATH) is None

DATA_REPRESENTATION_CONFIG = {
    "expression_source": {"kind": "X", "name": None},
    "normalize_strategy": "auto",
    "log1p": True,
    "remove_zero_count_cells": True,
    "spatial_source": {"kind": "obsm", "key": "spatial"},
    "apply_hvg_selection": APPLY_HVG_SELECTION,
}

base_config = {
    "data_path_main": DATA_ROOT,
    "output_dir": PROCESSED_DATA_DIR,
    "adata_folder_name": ADATA_FOLDER_NAME,
    "ligand_receptor_filedir_cellchatdb": CELLCHAT_DB,
    "ligand_receptor_filedir_scSeqComm": SCSEQCOMM_DB,
    "sample_col": SAMPLE_COL,
    "sample_name_obs_col": "source_file_prefix",
    "sample_attr_obs_col": SAMPLE_COL,
    "cell_class_col": CELL_CLASS_COL,
    "pyg_obs_fields": PYG_EXTRA_OBS_FIELDS,
    "n_hvg": N_HVG,
    "n_hvg_lr": N_HVG_LR,
    "num_neighbors": NUM_NEIGHBORS,
    "per_file_hook": pancancer_per_file_hook,
    "apply_lr_corr_filter": APPLY_LR_CORR_FILTER,
    "skip_lr_filter_when_predefined": True,
    **DATA_REPRESENTATION_CONFIG,
}

lr_list_path_use = normalize_optional_path_local(LR_LIST_PATH)
gene_list_path_use = normalize_optional_path_local(GENE_LIST_PATH)

if lr_list_path_use is not None:
    base_config["lr_list_path"] = lr_list_path_use

if gene_list_path_use is not None:
    base_config["gene_list_path"] = gene_list_path_use

base_config


{'data_path_main': WindowsPath('D:/SpiderNet/Data/Pancancer'),
 'output_dir': WindowsPath('D:/SpiderNet/Results/Pancancer/ProcessedData'),
 'adata_folder_name': 'adata',
 'ligand_receptor_filedir_cellchatdb': WindowsPath('D:/SpiderNet/SpiderNet_proj/SpiderNet_Project/SpiderNet/resources/Human_LR_pairs_Cellchatdb.csv'),
 'ligand_receptor_filedir_scSeqComm': WindowsPath('D:/SpiderNet/SpiderNet_proj/SpiderNet_Project/SpiderNet/resources/Human_LR_pairs_scSeqComm.csv'),
 'sample_col': 'SampleID',
 'sample_name_obs_col': 'source_file_prefix',
 'sample_attr_obs_col': 'SampleID',
 'cell_class_col': 'celltype_final',
 'pyg_obs_fields': {},
 'n_hvg': 1000,
 'n_hvg_lr': 2000,
 'num_neighbors': 5,
 'per_file_hook': <function __main__.pancancer_per_file_hook(adata, context)>,
 'apply_lr_corr_filter': True,
 'skip_lr_filter_when_predefined': True,
 'expression_source': {'kind': 'X', 'name': None},
 'normalize_strategy': 'auto',
 'log1p': True,
 'remove_zero_count_cells': True,
 'spatial_source': {'k

### Check paths and AnnData fields

Use the next cell to confirm that paths exist and the first `.h5ad` file has the expected fields.


In [5]:
adata_dir = Path(DATA_ROOT) / ADATA_FOLDER_NAME
sample_files = sorted(adata_dir.glob("*.h5ad"))
sample_path = sample_files[0] if sample_files else None

rows = [
    {"item": "DATA_ROOT", "value": str(DATA_ROOT), "ok": Path(DATA_ROOT).exists()},
    {"item": "OUTPUT_ROOT", "value": str(OUTPUT_ROOT), "ok": Path(OUTPUT_ROOT).exists()},
    {"item": "PROCESSED_DATA_DIR", "value": str(PROCESSED_DATA_DIR), "ok": Path(PROCESSED_DATA_DIR).exists()},
    {"item": "CellChat DB", "value": str(CELLCHAT_DB), "ok": Path(CELLCHAT_DB).exists()},
    {"item": "scSeqComm DB", "value": str(SCSEQCOMM_DB), "ok": Path(SCSEQCOMM_DB).exists()},
    {"item": "adata folder", "value": str(adata_dir), "ok": adata_dir.exists()},
    {"item": "number of .h5ad files", "value": len(sample_files), "ok": len(sample_files) > 0},
]

if sample_path is not None:
    adata_example = sc.read_h5ad(sample_path, backed="r")
    rows.extend([
        {"item": "example file", "value": sample_path.name, "ok": True},
        {"item": "obs['subslice_id']", "value": "subslice_id", "ok": "subslice_id" in adata_example.obs.columns},
        {"item": f"obs['{CELL_CLASS_COL}']", "value": CELL_CLASS_COL, "ok": CELL_CLASS_COL in adata_example.obs.columns},
        {"item": f"obsm['{SPATIAL_KEY}']", "value": SPATIAL_KEY, "ok": SPATIAL_KEY in adata_example.obsm_keys()},
        {"item": f"obs['{SAMPLE_COL}']", "value": "created during preprocessing", "ok": True},
    ])
    adata_example.file.close()
else:
    rows.append({"item": "example file", "value": "No .h5ad file found", "ok": False})

display(pd.DataFrame(rows))


C:\Users\junji\AppData\Local\Temp\ipykernel_17336\1286039046.py:21: FutureWarning: Use obsm (e.g. `k in adata.obsm` or `adata.obsm.keys() | {'u'}`) instead of AnnData.obsm_keys, AnnData.obsm_keys is deprecated and will be removed in the future.
  {"item": f"obsm['{SPATIAL_KEY}']", "value": SPATIAL_KEY, "ok": SPATIAL_KEY in adata_example.obsm_keys()},


,item,value,ok
0,DATA_ROOT,D:\SpiderNet\Data\Pancancer,True
1,OUTPUT_ROOT,D:\SpiderNet\Results\Pancancer,True
2,PROCESSED_DATA_DIR,D:\SpiderNet\Results\Pancancer\ProcessedData,True
3,CellChat DB,D:\SpiderNet\SpiderNet_proj\SpiderNet_Project\...,True
4,scSeqComm DB,D:\SpiderNet\SpiderNet_proj\SpiderNet_Project\...,True
5,adata folder,D:\SpiderNet\Data\Pancancer\adata,True
6,number of .h5ad files,40,True
7,example file,HumanBreastCancerPatient1_subslice_0_annotated...,True
8,obs['subslice_id'],subslice_id,True
9,obs['celltype_final'],celltype_final,True


## 3. Build preprocessing and training configs

These values are taken from the dataset setup cell.


In [6]:
preprocess_cfg = PreprocessConfig(
    n_hvg=N_HVG,
    n_hvg_lr=N_HVG_LR,
    num_neighbors=NUM_NEIGHBORS,
)

train_cfg = TrainingConfig(
    dim_envir=DIM_ENVIR,
    n_jobs=N_JOBS,
    max_epoch=MAX_EPOCH,
    version=VERSION,
)

run_dirs = paths.ensure_dirs(
    dim_envir=train_cfg.dim_envir,
)

processed_data_dir = Path(PROCESSED_DATA_DIR)
processed_data_dir.mkdir(parents=True, exist_ok=True)

print("Processed data directory:", processed_data_dir)
print("Model/result run directories:", run_dirs)

with open(paths.output_root / "run_dirs.json", "w", encoding="utf-8") as handle:
    json.dump({k: str(v) for k, v in run_dirs.items()}, handle, indent=2)

with open(paths.output_root / "processed_data_dir.json", "w", encoding="utf-8") as handle:
    json.dump({"processed_data_dir": str(processed_data_dir)}, handle, indent=2)


Processed data directory: D:\SpiderNet\Results\Pancancer\ProcessedData
Model/result run directories: {'run_dir': WindowsPath('D:/SpiderNet/Results/Pancancer/V1/SpiderNet_Result_dim11'), 'model_dir': WindowsPath('D:/SpiderNet/Results/Pancancer/V1/SpiderNet_Result_dim11/Model')}


In [7]:
preprocess_cfg

PreprocessConfig(n_hvg=1000, n_hvg_lr=2000, num_neighbors=5)

## 4. Preprocess raw data and load processed data

This notebook now uses `SpiderNet.dataloading_unified.prepare_processed_bundle_unified`, with the Pancancer-specific hook from `spidernet_dataloading_MIdimselection_Pancancer`.

Processed objects are saved in `PROCESSED_DATA_DIR`; model outputs are saved separately in `run_dirs["run_dir"]`.


In [8]:
if RUN_PREPROCESSING:
    dataloading_config = dict(base_config)

    if APPLY_LR_CORR_FILTER:
        if LR_CORR_THRESHOLD is None:
            preview = preview_lr_corr_distribution(base_config, show_plot=True)
            if preview.get("plot_path") is not None:
                print("LR-correlation preview saved to:", preview["plot_path"])
            raise ValueError(
                "LR_CORR_THRESHOLD is None. Inspect the preview plot, set LR_CORR_THRESHOLD, "
                "and rerun this cell."
            )
        dataloading_config["lr_corr_threshold"] = float(LR_CORR_THRESHOLD)

    bundle = prepare_processed_bundle_unified(dataloading_config)

    # Save one row per processed Pancancer batch. This file is used by optional downstream summaries.
    obs = bundle["adata"].obs.copy()
    metadata_sample = (
        obs.groupby(SAMPLE_COL, dropna=False)
          .agg(
              source_file_prefix=("source_file_prefix", "first"),
              subslice_id=("subslice_id", "first"),
              num_cells=(SAMPLE_COL, "size"),
              n_cell_types=(CELL_CLASS_COL, pd.Series.nunique),
          )
          .reset_index()
    )

    celltype_counts = pd.crosstab(obs[SAMPLE_COL], obs[CELL_CLASS_COL]).reset_index()
    celltype_counts.columns = [SAMPLE_COL] + [f"n_{col}" for col in celltype_counts.columns[1:]]

    metadata_sample = metadata_sample.merge(celltype_counts, on=SAMPLE_COL, how="left")
    metadata_sample = metadata_sample.sort_values(
        ["source_file_prefix", "subslice_id", SAMPLE_COL]
    ).reset_index(drop=True)

    metadata_sample_path = processed_data_dir / "metadata_sample.csv"
    metadata_sample.to_csv(metadata_sample_path, index=False)

    print("Processed bundle saved to:", bundle["output_dir"])
    print("Metadata saved to:", metadata_sample_path)
    display(metadata_sample.head())
else:
    print("Skipping preprocessing.")
    print("Using existing processed data directory:", processed_data_dir)


Step 1: Load the data and perform preprocessing
Reading AnnData files from: D:\SpiderNet\Data\Pancancer\adata
HumanBreastCancerPatient1_subslice_0_annotated.h5ad: remove 65 zero-count cells
AnnData object with n_obs × n_vars = 47423 × 500
    obs: 'fov', 'volume', 'center_x', 'center_y', 'min_x', 'max_x', 'min_y', 'max_y', 'total_counts', 'celltype_final', 'subslice_id', 'CELL_TYPE', 'source_file_prefix', 'SampleID'
    obsm: 'spatial'
HumanBreastCancerPatient1_subslice_1_annotated.h5ad: remove 301 zero-count cells
HumanBreastCancerPatient1_subslice_2_annotated.h5ad: remove 260 zero-count cells
HumanBreastCancerPatient1_subslice_3_annotated.h5ad: remove 214 zero-count cells
HumanBreastCancerPatient1_subslice_4_annotated.h5ad: remove 140 zero-count cells
HumanColonCancerPatient2_subslice_0_annotated.h5ad: remove 364 zero-count cells
HumanColonCancerPatient2_subslice_1_annotated.h5ad: remove 222 zero-count cells
HumanColonCancerPatient2_subslice_2_annotated.h5ad: remove 144 zero-count ce

D:\SpiderNet\SpiderNet_proj\SpiderNet_Project\SpiderNet\dataloading_unified.py:996: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = sc.AnnData.concatenate(*adata_file_list, batch_key="_concat_batch", index_unique=None)


Number of ligand-receptor pairs from CellChatDB: 210
Total number of Ligand–Receptor pairs after direct CellChat selection and branch-local deduplication: 149
Step 2: Build sample-level SpiderNet data dictionaries
Step 3: Build spatial neighbor graph
Total number of edges across all samples: 6201650
Step 4: One-hot encoding of cell types and PyG object creation
Step 5: Compute cell–cell–LRpair tensor
Step 6: Filter LR pairs based on coverage across edges
Quantiles of LR pair activation proportion: [0.00240275 0.01324174 0.06194664 0.15095965 0.77253085]
Step 7: Compute pairwise LR correlation across samples
Step 8: Finalize PyG features and cell labels
Step 9: Compute neighboring cell-type proportions
Step 11: Build batch-aligned adata_list
Step 12: Save processed objects


C:\Users\junji\miniconda3\envs\SpiderNet_env\Lib\site-packages\anndata\_io\utils.py:272: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


Done.


C:\Users\junji\AppData\Local\Temp\ipykernel_17336\940381948.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs.groupby(SAMPLE_COL, dropna=False)


Processed bundle saved to: D:\SpiderNet\Results\Pancancer\ProcessedData
Metadata saved to: D:\SpiderNet\Results\Pancancer\ProcessedData\metadata_sample.csv


,SampleID,source_file_prefix,subslice_id,num_cells,n_cell_types,n_B cell,n_Breast-cancercell,n_CD4_T,n_CD8_T/NK,n_Colon-cancercell,...,n_Liver-cancercell,n_Lung-cancercell,n_Macrophage,n_Mast cell,n_Melanoma-cancercell,n_Ovarian-cancercell,n_Prostate-cancercell,n_Treg,n_Uterine-cancercell,n_low_exp
0,HumanBreastCancerPatient1_subslice0,HumanBreastCancerPatient1,0,47423,11,236,26090,619,1184,0,...,0,0,3985,24,0,0,0,197,0,8
1,HumanBreastCancerPatient1_subslice1,HumanBreastCancerPatient1,1,40606,11,724,23753,579,1207,0,...,0,0,3204,27,0,0,0,212,0,4
2,HumanBreastCancerPatient1_subslice2,HumanBreastCancerPatient1,2,42502,11,256,18110,839,1517,0,...,0,0,4835,38,0,0,0,206,0,3
3,HumanBreastCancerPatient1_subslice3,HumanBreastCancerPatient1,3,31827,11,396,14044,856,1155,0,...,0,0,3419,22,0,0,0,167,0,1
4,HumanBreastCancerPatient1_subslice4,HumanBreastCancerPatient1,4,37125,11,198,20042,447,872,0,...,0,0,3226,52,0,0,0,177,0,6


Reload processed objects from `PROCESSED_DATA_DIR` for training and downstream analysis.

Check that the number of batches, LR pairs, and training genes is reasonable.


In [9]:
from SpiderNet.io import load_processed_data

required_processed_files = [
    "adata_all.h5ad",
    "adata_list.pkl",
    "SpiderNet_data_pyg_list.pkl",
    "LR_list.pkl",
    "genenames_train.pkl",
]

missing_processed_files = [
    f for f in required_processed_files
    if not (processed_data_dir / f).exists()
]

if missing_processed_files:
    raise FileNotFoundError(
        f"ProcessedData is incomplete. Missing files under {processed_data_dir}: "
        f"{missing_processed_files}. Run the unified preprocessing cell above first."
    )

processed = load_processed_data(processed_data_dir)

print("Processed data directory:", processed_data_dir)
print("Number of batches:", len(processed.spidernet_data))
print("Number of LR pairs:", len(processed.lr_list))
print("Number of training genes:", processed.genenames_train.shape[0])


Processed data directory: D:\SpiderNet\Results\Pancancer\ProcessedData
Number of batches: 40
Number of LR pairs: 106
Number of training genes: 500


Check whether the processed data are non-empty and internally consistent before training.


In [10]:
summary_rows = []
for i in range(len(processed.adata_list)):
    adata_i = processed.adata_list[i]
    edge_index_max = torch.max(processed.spidernet_data[i]["edge_index"]).cpu().item()
    summary_rows.append(
        {
            "batch_index": i,
            "n_cells": adata_i.n_obs,
            "n_genes": adata_i.n_vars,
            "edge_index_max": edge_index_max,
            "edge_index_matches_n_cells": adata_i.n_obs == (edge_index_max + 1),
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

issues = []
if len(processed.spidernet_data) == 0:
    issues.append("No processed batches were loaded.")
if len(processed.lr_list) == 0:
    issues.append("No ligand-receptor pairs were retained.")
if processed.genenames_train.shape[0] == 0:
    issues.append("No training genes were retained.")
if not summary_df["edge_index_matches_n_cells"].all():
    issues.append("At least one batch has an edge index / cell-count mismatch.")

if issues:
    print("Sanity check flagged the following issues:")
    for issue in issues:
        print("-", issue)
else:
    print("Sanity checks passed. The processed data appear internally consistent.")


,batch_index,n_cells,n_genes,edge_index_max,edge_index_matches_n_cells
0,0,47423,500,47422,True
1,1,40606,500,40605,True
2,2,42502,500,42501,True
3,3,31827,500,31826,True
4,4,37125,500,37124,True
5,5,45114,500,45113,True
6,6,34468,500,34467,True
7,7,53402,500,53401,True
8,8,37891,500,37890,True
9,9,39533,500,39532,True


Sanity checks passed. The processed data appear internally consistent.


## 5. Build and train the SpiderNet model

Initialize the model using the processed data and training settings.


In [11]:
from SpiderNet.api import build_model
model = build_model(
    processed=processed,
    train_cfg=train_cfg,
    device=device,
)

print(model)


Number of GPUs available: 1
GPU 0: NVIDIA GeForce RTX 4070
SpiderNet_model(
  (dropout_fun): Dropout(p=0.1, inplace=False)
  (Relu): ReLU()
  (Sigmoid): Sigmoid()
  (enc_factor_envir_pre_receiver): Sequential(
    (0): Linear(in_features=500, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): Tanh()
  )
  (enc_factor_envir_pre_sender): Sequential(
    (0): Linear(in_features=500, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): Tanh()
  )
  (enc_factor_envir): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=11, bias=True)
  )
)


Runs model training and saves checkpoints.

In [12]:
from SpiderNet.api import run_training
model = run_training(
    model=model,
    processed=processed,
    train_cfg=train_cfg,
    model_dir=run_dirs["model_dir"],
    device=device,
)


Loading existing trained model...


### Extract the inferred meta-interactions and loadings

This step generates the main outputs:
- MI activities
- LR loadings
- sender loadings
- receiver loadings


In [13]:
from SpiderNet.api import infer_meta_interactions, normalize_outputs
results = infer_meta_interactions(model=model, processed=processed)
results = normalize_outputs(results)

print("Factor_envir shape:", results["factor_envir"].shape)
print("LR loading shape:", results["loading_lr"].shape)
print("Receiver loading shape:", results["loading_receiver"].shape)
print("Sender loading shape:", results["loading_sender"].shape)


Factor_envir shape: (6201650, 11)
LR loading shape: (11, 106)
Receiver loading shape: (11, 500)
Sender loading shape: (11, 500)


Saves the main outputs and config files.

In [14]:
from SpiderNet.api import export_results

export_results(
    results=results,
    processed=processed,
    output_dir=run_dirs["run_dir"],
)

# Save the model, preprocessing, and data-loading configurations for reproducibility.
with open(run_dirs["model_dir"] / "SpiderNet_model_config.json", "w", encoding="utf-8") as handle:
    json.dump(train_cfg.to_dict(), handle, indent=2)

with open(run_dirs["model_dir"] / "SpiderNet_preprocess_config.json", "w", encoding="utf-8") as handle:
    json.dump(preprocess_cfg.to_dict(), handle, indent=2)

dataloading_config_export = dict(base_config)
dataloading_config_export["lr_corr_threshold"] = LR_CORR_THRESHOLD
dataloading_config_export["processed_data_dir"] = str(processed_data_dir)
dataloading_config_export["run_dir"] = str(run_dirs["run_dir"])

def json_safe_for_export(value):
    if callable(value):
        return getattr(value, "__name__", str(value))
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(k): json_safe_for_export(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe_for_export(v) for v in value]
    return value

with open(run_dirs["model_dir"] / "SpiderNet_dataloading_config.json", "w", encoding="utf-8") as handle:
    json.dump(json_safe_for_export(dataloading_config_export), handle, indent=2)

print(f"Results saved to: {run_dirs['run_dir']}")
print(f"Processed objects loaded from: {processed_data_dir}")


TypeError: export_results() missing 1 required positional argument: 'precessed_data_dir'

## Key output files

Model outputs are saved in `run_dirs["run_dir"]`:
- `Factor_envir_use.npy`
- `loading_LR_use.npy`
- `loading_sender_use.npy`
- `loading_receiver_use.npy`

Processed objects are saved in `PROCESSED_DATA_DIR`:
- `adata_all.h5ad`
- `adata_list.pkl`
- `SpiderNet_data_pyg_list.pkl`
- `LR_list.pkl`
- `batch_cell_unique.pkl`
- `metadata_sample.csv`


## Optional downstream analyses

The remaining sections help interpret results, but are not required to run SpiderNet on a new dataset.


## Optional 1. MI correlation analysis

Explores relationships among inferred meta-interactions.


In [ ]:
from SpiderNet.analysis import MI_correlation

mi_results = MI_correlation(run_dirs['run_dir'], run_dirs['run_dir'] / "Factor_envir_use.npy", show=True)


## Optional 2. LR loading-based pathway enrichment

Interprets each MI using ligand--receptor pathway content.


In [ ]:
from SpiderNet.analysis import LRLoading_enrichment

LRLoading_enrichment(
    loading_LR_use_path=run_dirs["run_dir"] / "loading_LR_use.npy",
    lr_list_path=processed_data_dir / "LR_list.pkl",
    lr_list_cellchatdb_path=processed_data_dir / "LR_list_cellchatdb.pkl",
    lr_meta_cellchatdb_path=processed_data_dir / "LR_meta_cellchatdb.pkl",
    Factor_envir_use_path=run_dirs["run_dir"] / "Factor_envir_use.npy",
    file_savepath_main=run_dirs["run_dir"],
    show=True,
)


## Optional 3. Sender-cell-type--receiver-cell-type interaction analysis

Summarizes MI enrichment across sender--receiver cell-type pairs.


In [ ]:
from SpiderNet.analysis import MI_Celltypepair_enrichment

metadata_sample_path = processed_data_dir / "metadata_sample.csv"
metadata_sample_path_use = metadata_sample_path if metadata_sample_path.exists() else "None"

MI_Celltypepair_enrichment(
    SpiderNet_data_pyg_list_path=processed_data_dir / "SpiderNet_data_pyg_list.pkl",
    Factor_envir_use_path=run_dirs["run_dir"] / "Factor_envir_use.npy",
    file_savepath_main=run_dirs["run_dir"],
    metadata_sample_path=metadata_sample_path_use,
    LR_loading_pathway_path=run_dirs["run_dir"] / "LR_loading_pathway.csv",
    dim_envir=train_cfg.dim_envir,
    MIlevel_agg_threshold=0.6,
    batch_cell_unique_path=processed_data_dir / "batch_cell_unique.pkl",
    adata_list_path=processed_data_dir / "adata_list.pkl",
    adata_copy_path=processed_data_dir / "adata_all.h5ad",
    show=True,
)


## Adapting this notebook to your own data

Start by editing the dataset setup cell.

What usually changes:
- `DATA_ROOT`, `OUTPUT_ROOT`, `PROCESSED_DATA_DIR`, and `ADATA_FOLDER_NAME`
- species
- `SAMPLE_COL`, `CELL_CLASS_COL`, and `SPATIAL_KEY`
- the dataset-specific `per_file_hook`
- preprocessing parameters
- training parameters

Recommended order:
1. edit the dataset setup cell
2. run the path / AnnData check
3. run unified preprocessing or set `RUN_PREPROCESSING = False` if `PROCESSED_DATA_DIR` is already complete
4. load and sanity-check processed objects
5. train, infer MIs, and export model outputs


In [ ]:
# Example quick checks:
# processed.adata_list[0].obs.head()
# processed.spidernet_data[0]
